# Lab: Kokoro ← Stage 1–2 synthesis manifest

**Goal:** Check TTS compatibility with prepared pipeline output (`spoken_text`, `voice_profile_id`, multi-speaker segments).

**Not:** final model selection · production adapter · silent script edits

Runtime: Colab GPU (L4/A100 preferred).

## 0. Config — set your repo URL

In [ ]:
# @title Repo settings
REPO_URL = "https://github.com/phamtu2x5/ielts-script2audio.git"
BRANCH = "main"
WORKDIR = "/content/ielts-script2audio"


## 1. Clone repo

In [ ]:
import os
from pathlib import Path

if not Path(WORKDIR).exists():
    !git clone --branch {BRANCH} {REPO_URL} {WORKDIR}
else:
    %cd {WORKDIR}
    !git pull
%cd {WORKDIR}
print("CWD:", os.getcwd())

## 2. Install control-plane + Kokoro

Stage 1–2 package is local editable install. Kokoro needs `espeak-ng`.

In [ ]:
!pip -q install -e ".[dev]" kokoro soundfile
!apt-get -qq update && apt-get -qq install -y espeak-ng

import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

## 3. Stage 1–2: build synthesis manifest

This is the **same contract** production will use later.

In [ ]:
!mkdir -p outputs lab_audio/kokoro_part1
!ielts-s2a prepare examples/part1_valid.json --pretty -o outputs/part1_manifest.json

import json
from pathlib import Path

m = json.loads(Path("outputs/part1_manifest.json").read_text())
print("transcript_id:", m["transcript_id"])
print("valid:", m["validation"]["valid"])
print("n_requests:", len(m["requests"]))
print("voice profiles:", [(e["speaker_id"], e["voice_profile_id"]) for e in m["speaker_map"]["entries"]])
for u in m["prepared_utterances"]:
    if u["event_id"] in {"e004", "e006"}:
        print(u["event_id"], "display:", u["display_text"])
        print("       spoken :", u["spoken_text"])

## 4. Lab render (Kokoro consumes manifest)

Maps `voice_profile_id` → Kokoro fixed IDs via `configs/voice_maps/kokoro_en_gb_part1.json`.

Uses **`spoken_text`** (Stage 2), not raw script rewrite.

In [ ]:
!python scripts/lab_render_kokoro_from_manifest.py \
  --manifest outputs/part1_manifest.json \
  --voice-map configs/voice_maps/kokoro_en_gb_part1.json \
  --out-dir lab_audio/kokoro_part1

report = json.loads(Path("lab_audio/kokoro_part1/lab_render_report.json").read_text())
print("ok:", report["ok_count"], "fail:", report["fail_count"])
print("notes:", report["compatibility_notes"])
for s in report["segments"]:
    print(s["segment_id"], s["status"], s.get("backend_voice_id"), s.get("error"))

## 5. Listen + track từng câu (đối chiếu nội dung)

Mỗi hàng = **một segment**:

- **display_text** = chữ trên script (Stage 1, không được tự sửa)
- **spoken_text** = chữ đã chuẩn hóa để TTS đọc (Stage 2)
- **audio** = file vừa inference
- Bạn điền **content_match**: `yes` / `partial` / `no` — nghe có khớp nội dung không

Không cần nhớ file wav nằm đâu: bảng gắn đúng text với đúng audio.


In [ ]:
from pathlib import Path
from IPython.display import Audio, HTML, display
import json
import sys

sys.path.insert(0, str(Path(".").resolve()))
from scripts.lab_show_segment_tracking import build_rows, build_html, load_report

REPORT_PATH = Path("lab_audio/kokoro_part1/lab_render_report.json")
AUDIO_DIR = REPORT_PATH.parent

report = load_report(REPORT_PATH)
rows = build_rows(report, audio_dir=AUDIO_DIR)

# 1) Bảng HTML: text kỳ vọng + audio player từng dòng
display(HTML(build_html(rows, title=f"Tracking — {report.get('transcript_id')}")))

print("\n--- Chi tiết từng segment (copy/ghi chú) ---\n")
for row in rows:
    print("=" * 72)
    print(
        f"[{row['index']}] {row['segment_id']} | speaker={row['speaker_id']} | "
        f"voice={row['backend_voice_id']} | status={row['status']}"
    )
    print(f"  FILE    : {row['output_filename']}")
    print(f"  DISPLAY : {row['display_text']}")
    print(f"  SPOKEN  : {row['spoken_text']}")
    if row.get("protected_region_ids"):
        print(f"  protected: {row['protected_region_ids']}")
    wav = Path(row["wav_path"]) if row.get("wav_path") else None
    if wav and wav.is_file():
        display(Audio(filename=str(wav)))
    else:
        print("  (no wav file)")
    print("  → content_match?  yes / partial / no")
    print("  → notes:")


## 5b. Ghi nhận xét tracking (tuỳ chọn)

Sau khi nghe, sửa dict `reviews` bên dưới (`yes` / `partial` / `no`) rồi chạy cell kế tiếp để lưu file review.  
**Không** sửa `display_text` trong transcript — chỉ ghi match/notes.


In [ ]:
from pathlib import Path
import json

# Key = segment_id (đúng id trong report). Value = đánh giá sau khi nghe.
reviews = {
    # "seg_0001": {"content_match": "yes", "notes": ""},
    # "seg_0002": {"content_match": "yes", "notes": ""},
    # "seg_0003": {"content_match": "yes", "notes": ""},
    # "seg_0004": {"content_match": "partial", "notes": "spelling hơi dính"},
    # "seg_0005": {"content_match": "yes", "notes": ""},
    # "seg_0006": {"content_match": "no", "notes": "postcode không rõ"},
}

report_path = Path("lab_audio/kokoro_part1/lab_render_report.json")
report = json.loads(report_path.read_text(encoding="utf-8"))

filled = []
for seg in report.get("segments") or []:
    sid = seg.get("segment_id")
    human = reviews.get(sid) or {}
    filled.append(
        {
            "segment_id": sid,
            "event_id": seg.get("event_id"),
            "speaker_id": seg.get("speaker_id"),
            "backend_voice_id": seg.get("backend_voice_id"),
            "output_filename": seg.get("output_filename"),
            "display_text": seg.get("display_text"),
            "spoken_text": seg.get("spoken_text"),
            "text_fed_to_tts": seg.get("text_fed_to_tts"),
            "status": seg.get("status"),
            "content_match": human.get("content_match", ""),
            "notes": human.get("notes", ""),
        }
    )

out = Path("lab_audio/kokoro_part1/segment_review_filled.json")
out.write_text(json.dumps(filled, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"Wrote {out}")
for row in filled:
    print(row["segment_id"], "match=", row["content_match"] or "(empty)", "|", (row["display_text"] or "")[:50])


## 6. Optional ablation: feed display_text instead of spoken_text

If this sounds **worse** on spelling/postcode, Stage-2 normalisation is doing useful work.

In [ ]:
!python scripts/lab_render_kokoro_from_manifest.py \
  --manifest outputs/part1_manifest.json \
  --voice-map configs/voice_maps/kokoro_en_gb_part1.json \
  --out-dir lab_audio/kokoro_part1_display_text \
  --use-display-text \
  --limit 6

## 7. Human checklist

Open `docs/research/lab/checklist-stage1-2-compatibility.md` and mark:

1. Two speakers distinguishable?
2. Spelling (e004) clear with spoken_text?
3. Postcode (e006) usable?
4. Any crash / empty audio?
5. Did we need to edit the transcript script? (**must be no** — report issues instead)

Verdict: **Pass lab / Borderline / Fail lab** — still **not** final TTS selection.